In [ ]:
%matplotlib inline

# Hyperparameter Tuning Script for U-FNO model, used to find optimal width and modes

In [ ]:
import os
from datetime import datetime 
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

from data_utils import prepare_dataloaders
from U_FNO import UFNO3d

# User Settings
in_channels = 6
out_channels = 2
BATCH_SIZE = 2
NORMALIZATION_METHOD = 'z-score'  # 'z-score' or 'min-max'
INPUT_HDF5 = 'small_input_arrays.hdf5'
OUTPUT_HDF5 = 'small_output_arrays.hdf5'
STATS_PATH = 'normalization_stats'

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

training_loader, validation_loader, test_loader = prepare_dataloaders(
    input_hdf5=INPUT_HDF5,
    output_hdf5=OUTPUT_HDF5,
    batch_size=BATCH_SIZE,
    norm_method=NORMALIZATION_METHOD,
    stats_path=STATS_PATH
)

print(f"DataLoaders are ready. Train size: {len(training_loader.dataset)}")

In [ ]:
def objective(trial):
    # Define the hyperparameter search space to match the UFNO3d model
    params = {
        'mode1': trial.suggest_int('mode1', 2, 16, step=2),
        'mode2': trial.suggest_int('mode2', 2, 16, step=2),
        'mode3': trial.suggest_int('mode3', 2, 6, step=2),
        'width': trial.suggest_categorical('width', [16, 32, 64]),
        'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    }

    # Instantiate your UFNO3d model with the trial's parameters
    model = UFNO3d(
        modes1=params['mode1'],
        modes2=params['mode2'],
        modes3=params['mode3'],
        width=params['width'],
        in_channels=in_channels,
        out_channels=out_channels,
        UNet=True 
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=params['lr'])
    loss_fn = nn.MSELoss()
    accumulation_steps = 4
    
    epochs_per_trial = 50
    best_vloss = float('inf')
    best_epoch = -1

    for epoch in range(epochs_per_trial):
        model.train()
        optimizer.zero_grad()
        for i, data in enumerate(training_loader):
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels) / accumulation_steps
            loss.backward()
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(training_loader):
                optimizer.step()
                optimizer.zero_grad()
        
        model.eval()
        running_vloss = 0.0
        with torch.no_grad():
            for vdata in validation_loader:
                vinputs, vlabels = vdata[0].to(device), vdata[1].to(device)
                voutputs = model(vinputs)
                vloss = loss_fn(voutputs, vlabels)
                running_vloss += vloss.item()
        
        avg_vloss = running_vloss / len(validation_loader)
        
        if avg_vloss < best_vloss:
            best_vloss = avg_vloss
            best_epoch = epoch + 1
            
        trial.report(avg_vloss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    trial.set_user_attr("best_epoch", best_epoch)
    return best_vloss

# Start tuning
print("\n--- Starting Hyperparameter Tuning ---")
study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=30) 


print("\n Hyperparameter tuning finished.")

results_df = study.trials_dataframe()
best_epochs = [t.user_attrs.get("best_epoch", -1) for t in study.trials]
results_df['best_epoch'] = best_epochs

# Define columns to display
cols = [
    'number', 'value', 'best_epoch', 'state', 'params_mode1', 'params_mode2',
    'params_mode3', 'params_width', 'params_lr', 'params_accumulation_steps'
]
results_df = results_df[cols].rename(columns={'value': 'best_validation_loss'})

print("\n--- Results DataFrame ---")
print(results_df.sort_values(by='best_validation_loss', ascending=True).to_string())

# best model is grabbed here
best_params = study.best_trial.params
print("\n--- Best Hyperparameters ---")
print(best_params)

# Instantiate the final best model with the optimal hyperparameters
best_model = UFNO3d(
    mode1=best_params['mode1'],
    mode2=best_params['mode2'],
    mode3=best_params['mode3'],
    width=best_params['width'],
    in_channels=in_channels,
    out_channels=out_channels,
    UNet=True
)

print("\nBest model with the following architecture has been returned:")
print(best_model)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
model_path = f"best_model_{timestamp}.pt"
torch.save(best_model.state_dict(), model_path)
print(f"\n Best model's state_dict saved to: {model_path}")